<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/04_vision/42_vgg_resnet_inception_fractalnet.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# VGG, ResNet, Inception y FractalNet

**Pregunta guía:** ¿Cómo permiten los caminos de información entrenar redes profundas?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere TensorFlow.** ImageNet es un dataset y una competición, no una
arquitectura. VGG apila convoluciones 3×3; Inception procesa varias
escalas en paralelo; ResNet aprende un residuo $F(x)$ y suma $x+F(x)$;
FractalNet construye múltiples caminos auto-similares. Aquí inspeccionamos
bloques y tamaños; el notebook siguiente aborda transferencia.


In [ ]:
import gc
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow", tf.__version__)

def bloque_residual(x, filtros):
    atajo = x
    y = layers.Conv2D(filtros,3,padding="same",use_bias=False)(x)
    y = layers.BatchNormalization()(y); y = layers.ReLU()(y)
    y = layers.Conv2D(filtros,3,padding="same",use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if x.shape[-1] != filtros: atajo = layers.Conv2D(filtros,1)(atajo)
    return layers.ReLU()(layers.Add()([atajo,y]))

def bloque_inception(x, filtros=16):
    r1 = layers.Conv2D(filtros,1,padding="same",activation="relu")(x)
    r3 = layers.Conv2D(filtros,3,padding="same",activation="relu")(x)
    r5 = layers.Conv2D(filtros,5,padding="same",activation="relu")(x)
    rp = layers.MaxPool2D(3,strides=1,padding="same")(x)
    rp = layers.Conv2D(filtros,1,activation="relu")(rp)
    return layers.Concatenate()([r1,r3,r5,rp])

def bloque_fractal(x, filtros, profundidad):
    directo = layers.Conv2D(filtros,3,padding="same",activation="relu")(x)
    if profundidad == 1: return directo
    largo = bloque_fractal(x,filtros,profundidad-1)
    largo = bloque_fractal(largo,filtros,profundidad-1)
    return layers.Average()([directo,largo])


In [ ]:
entrada = keras.Input((32,32,16))
bloques = {
    "residual": bloque_residual(entrada,16),
    "inception": bloque_inception(entrada,16),
    "fractal_C3": bloque_fractal(entrada,16,3),
}
for nombre,salida in bloques.items():
    modelo = keras.Model(entrada,salida,name=nombre)
    print(nombre, "salida", modelo.output_shape, "parámetros", modelo.count_params())


In [ ]:
fábricas = {
    "VGG16": lambda: keras.applications.VGG16(weights=None,include_top=False,input_shape=(96,96,3)),
    "ResNet50": lambda: keras.applications.ResNet50(weights=None,include_top=False,input_shape=(96,96,3)),
    "InceptionV3": lambda: keras.applications.InceptionV3(weights=None,include_top=False,input_shape=(96,96,3)),
}
filas=[]
for nombre,fábrica in fábricas.items():
    keras.backend.clear_session(); modelo=fábrica()
    salida=modelo(tf.zeros((1,96,96,3)),training=False)
    filas.append({"arquitectura":nombre,"parámetros":modelo.count_params(),"salida":str(tuple(salida.shape))})
    del modelo; gc.collect()
display(pd.DataFrame(filas).set_index("arquitectura"))


**Preguntas:** ¿por qué $x+F(x)$ ofrece una ruta de gradiente? ¿Qué costo
tiene concatenar ramas Inception? ¿En qué se distingue FractalNet de una
simple red residual? Compare número de parámetros, activaciones y FLOPs:
parámetros no son una medida suficiente de costo.
